In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [68]:
# Cell 1: Install kagglehub (if not present)
# !pip install kagglehub -q

import kagglehub
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB4, ResNet101, ResNet152
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import cv2
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


In [45]:
# Cell 2: Download dataset using kagglehub
print("Downloading Deep Fake Dataset...")
path = kagglehub.dataset_download("ucimachinelearning/deep-fake-dataset")
print("Dataset downloaded to:", path)

# List files
for root, dirs, files in os.walk(path):
    print(root, "->", len(files), "files")

Dataset downloaded to: /kaggle/input/datasets/ucimachinelearning/deep-fake-dataset
/kaggle/input/datasets/ucimachinelearning/deep-fake-dataset -> 0 files
/kaggle/input/datasets/ucimachinelearning/deep-fake-dataset/dataset5 -> 1 files
/kaggle/input/datasets/ucimachinelearning/deep-fake-dataset/dataset5/fake -> 700 files
/kaggle/input/datasets/ucimachinelearning/deep-fake-dataset/dataset5/real -> 589 files


In [62]:
# Cell 3: Load CSV and create file paths with correct extension
import os
import pandas as pd
import numpy as np

csv_path = os.path.join(path, "dataset5", "data.csv")
df = pd.read_csv(csv_path)
print("DataFrame shape:", df.shape)
print(df.head())

# Map string labels to integers (fake → 0, real → 1)
df['label'] = df['label'].map({'fake': 0, 'real': 1})

# Base directory
base_dir = os.path.join(path, "dataset5")

# Construct filepath: base_dir / subfolder / images_id + '.jpg'
df['filepath'] = df.apply(
    lambda row: os.path.join(
        base_dir,
        'fake' if row['label'] == 0 else 'real',
        row['images_id'] + '.jpg'
    ),
    axis=1
)

# Verify first few paths
print("\nFirst 5 filepaths:")
print(df['filepath'].head())
print("\nFirst file exists?", os.path.exists(df['filepath'].iloc[0]))

# Check for missing files
missing = [not os.path.exists(fp) for fp in df['filepath']]
print(f"\nMissing files: {sum(missing)}")

# Label distribution
print("\nLabel distribution:")
print(df['label'].value_counts())

DataFrame shape: (1289, 2)
  images_id label
0    real_1  real
1   real_10  real
2  real_100  real
3  real_101  real
4  real_102  real

First 5 filepaths:
0    /kaggle/input/datasets/ucimachinelearning/deep...
1    /kaggle/input/datasets/ucimachinelearning/deep...
2    /kaggle/input/datasets/ucimachinelearning/deep...
3    /kaggle/input/datasets/ucimachinelearning/deep...
4    /kaggle/input/datasets/ucimachinelearning/deep...
Name: filepath, dtype: object

First file exists? True

Missing files: 0

Label distribution:
label
0    700
1    589
Name: count, dtype: int64


In [63]:
# Cell 4: Stratified split
X = df['filepath'].values
y = df['label'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)} samples")
print(f"Validation: {len(X_val)} samples")
print(f"Test: {len(X_test)} samples")
print(f"Train distribution: {np.bincount(y_train)}")
print(f"Val distribution:   {np.bincount(y_val)}")
print(f"Test distribution:  {np.bincount(y_test)}")

Train: 902 samples
Validation: 193 samples
Test: 194 samples
Train distribution: [490 412]
Val distribution:   [105  88]
Test distribution:  [105  89]


In [64]:
# Prepare DataFrames with string labels for flow_from_dataframe
train_df = pd.DataFrame({
    'filename': X_train,
    'class': ['fake' if label == 0 else 'real' for label in y_train]
})
val_df = pd.DataFrame({
    'filename': X_val,
    'class': ['fake' if label == 0 else 'real' for label in y_val]
})
test_df = pd.DataFrame({
    'filename': X_test,
    'class': ['fake' if label == 0 else 'real' for label in y_test]
})

train_gen = train_datagen.flow_from_dataframe(
    train_df, directory='/', x_col='filename', y_col='class',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', classes=['fake', 'real'], shuffle=True
)
val_gen = val_test_datagen.flow_from_dataframe(
    val_df, directory='/', x_col='filename', y_col='class',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', classes=['fake', 'real'], shuffle=False
)
test_gen = val_test_datagen.flow_from_dataframe(
    test_df, directory='/', x_col='filename', y_col='class',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', classes=['fake', 'real'], shuffle=False
)

steps_per_epoch = len(train_gen)
validation_steps = len(val_gen)
test_steps = len(test_gen)
print(f"steps_per_epoch = {steps_per_epoch}")

Found 902 validated image filenames belonging to 2 classes.
Found 193 validated image filenames belonging to 2 classes.
Found 194 validated image filenames belonging to 2 classes.
steps_per_epoch = 29


In [65]:
# --- Generator Diagnostics ---
print("=== Generator Diagnostics ===")
print(f"train_gen.samples = {train_gen.samples}")
print(f"val_gen.samples   = {val_gen.samples}")
print(f"test_gen.samples  = {test_gen.samples}")
print(f"steps_per_epoch = {steps_per_epoch} (should be > 0)")

# Try to fetch one batch from train_gen
try:
    x_batch, y_batch = next(train_gen)
    print(f"✅ train_gen batch: x.shape={x_batch.shape}, y.shape={y_batch.shape}")
except Exception as e:
    print(f"❌ train_gen error: {e}")

# Try to fetch one batch from val_gen
try:
    x_batch, y_batch = next(val_gen)
    print(f"✅ val_gen batch: x.shape={x_batch.shape}, y.shape={y_batch.shape}")
except Exception as e:
    print(f"❌ val_gen error: {e}")

=== Generator Diagnostics ===
train_gen.samples = 902
val_gen.samples   = 193
test_gen.samples  = 194
steps_per_epoch = 29 (should be > 0)
✅ train_gen batch: x.shape=(32, 224, 224, 3), y.shape=(32,)
✅ val_gen batch: x.shape=(32, 224, 224, 3), y.shape=(32,)


In [49]:
print("base_dir:", base_dir)
print("First 5 absolute paths:", X_train[:5])
print("First 5 relative paths:", X_train_rel[:5])
# Check if the first relative path exists when joined with base_dir
first_rel = X_train_rel[0]
print("Full path from rel + base_dir:", os.path.join(base_dir, first_rel))
print("Exists?", os.path.exists(os.path.join(base_dir, first_rel)))

base_dir: /kaggle/input/datasets/ucimachinelearning/deep-fake-dataset/dataset5
First 5 absolute paths: ['/kaggle/input/datasets/ucimachinelearning/deep-fake-dataset/dataset5/real/real_344'
 '/kaggle/input/datasets/ucimachinelearning/deep-fake-dataset/dataset5/real/fake_290'
 '/kaggle/input/datasets/ucimachinelearning/deep-fake-dataset/dataset5/real/fake_483'
 '/kaggle/input/datasets/ucimachinelearning/deep-fake-dataset/dataset5/real/real_45'
 '/kaggle/input/datasets/ucimachinelearning/deep-fake-dataset/dataset5/real/fake_98']
First 5 relative paths: ['real/real_344', 'real/fake_290', 'real/fake_483', 'real/real_45', 'real/fake_98']
Full path from rel + base_dir: /kaggle/input/datasets/ucimachinelearning/deep-fake-dataset/dataset5/real/real_344
Exists? False


In [50]:
real_dir = os.path.join(base_dir, 'real')
print("Files in real folder (first 10):", os.listdir(real_dir)[:10])

Files in real folder (first 10): ['real_332.jpg', 'real_541.jpg', 'real_111.jpg', 'real_523.jpg', 'real_430.jpg', 'real_377.jpg', 'real_539.jpg', 'real_490.jpg', 'real_161.jpg', 'real_59.jpg']


In [48]:
# Define the base directory where images are stored
base_dir = os.path.join(path, "dataset5")

# Convert absolute paths to relative paths (e.g., 'fake/fake_0001.jpg')
X_train_rel = [os.path.relpath(p, base_dir) for p in X_train]
X_val_rel   = [os.path.relpath(p, base_dir) for p in X_val]
X_test_rel  = [os.path.relpath(p, base_dir) for p in X_test]

# Create DataFrames with relative paths and string labels
train_df = pd.DataFrame({
    'filename': X_train_rel,
    'class': ['fake' if label == 0 else 'real' for label in y_train]
})
val_df = pd.DataFrame({
    'filename': X_val_rel,
    'class': ['fake' if label == 0 else 'real' for label in y_val]
})
test_df = pd.DataFrame({
    'filename': X_test_rel,
    'class': ['fake' if label == 0 else 'real' for label in y_test]
})

# Now use flow_from_dataframe with directory=base_dir
train_gen = train_datagen.flow_from_dataframe(
    train_df,
    directory=base_dir,           # <-- important
    x_col='filename',
    y_col='class',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=['fake', 'real'],
    shuffle=True
)

val_gen = val_test_datagen.flow_from_dataframe(
    val_df,
    directory=base_dir,
    x_col='filename',
    y_col='class',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=['fake', 'real'],
    shuffle=False
)

test_gen = val_test_datagen.flow_from_dataframe(
    test_df,
    directory=base_dir,
    x_col='filename',
    y_col='class',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=['fake', 'real'],
    shuffle=False
)

steps_per_epoch = len(train_gen)
validation_steps = len(val_gen)
test_steps = len(test_gen)

print(f"steps_per_epoch = {steps_per_epoch}")
print(f"validation_steps = {validation_steps}")
print(f"test_steps = {test_steps}")

Found 0 validated image filenames belonging to 2 classes.
Found 0 validated image filenames belonging to 2 classes.
Found 0 validated image filenames belonging to 2 classes.
steps_per_epoch = 0
validation_steps = 0
test_steps = 0


In [25]:
# Cell 5: Image parameters
IMG_SIZE = 224   # EfficientNet / ResNet default
BATCH_SIZE = 32

# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Only rescaling for validation/test
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Function to load and yield batches from file paths (to avoid storing all images in memory)
def flow_from_dataframe(filepaths, labels, datagen, batch_size=32, target_size=(IMG_SIZE, IMG_SIZE), shuffle=True):
    num_samples = len(filepaths)
    print(f"Generator initialized with {num_samples} samples")
    while True:
        indices = np.arange(num_samples)
        if shuffle:
            np.random.shuffle(indices)
        for start in range(0, num_samples, batch_size):
            end = min(start + batch_size, num_samples)
            batch_indices = indices[start:end]
            batch_paths = filepaths[batch_indices]
            batch_labels = labels[batch_indices]
            
            images = []
            for path in batch_paths:
                try:
                    img = tf.keras.preprocessing.image.load_img(path, target_size=target_size)
                    img = tf.keras.preprocessing.image.img_to_array(img)
                    images.append(img)
                except Exception as e:
                    print(f"Error loading {path}: {e}")
                    raise   # Stop and show error
            images = np.array(images, dtype=np.float32)
            
            # Apply augmentation
            flow = datagen.flow(images, batch_labels, batch_size=len(images))
            augmented_x, augmented_y = next(flow)
            yield augmented_x, augmented_y
# Create generators
train_generator = flow_from_dataframe(X_train, y_train, train_datagen, batch_size=BATCH_SIZE, shuffle=True)
val_generator = flow_from_dataframe(X_val, y_val, val_test_datagen, batch_size=BATCH_SIZE, shuffle=False)
test_generator = flow_from_dataframe(X_test, y_test, val_test_datagen, batch_size=BATCH_SIZE, shuffle=False)

# Steps per epoch
steps_per_epoch = len(X_train) // BATCH_SIZE
validation_steps = len(X_val) // BATCH_SIZE
test_steps = len(X_test) // BATCH_SIZE

In [26]:
print(y_train[:10])
print(y_train.dtype)

[1 0 0 1 0 0 0 0 1 0]
int64


In [27]:
steps_per_epoch = len(X_train) // BATCH_SIZE
print("steps_per_epoch =", steps_per_epoch)

steps_per_epoch = 28


In [28]:
# Create a temporary DataFrame with filepaths and labels
train_df = pd.DataFrame({'filename': X_train, 'class': y_train})
val_df   = pd.DataFrame({'filename': X_val,   'class': y_val})
test_df  = pd.DataFrame({'filename': X_test,  'class': y_test})

train_gen = train_datagen.flow_from_dataframe(
    train_df,
    x_col='filename',
    y_col='class',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

val_gen = val_test_datagen.flow_from_dataframe(
    val_df,
    x_col='filename',
    y_col='class',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_gen = val_test_datagen.flow_from_dataframe(
    test_df,
    x_col='filename',
    y_col='class',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

steps_per_epoch = len(train_gen)
validation_steps = len(val_gen)
test_steps = len(test_gen)

TypeError: If class_mode="binary", y_col="class" column values must be strings.

In [57]:
# Cell 6: CBAM layer
def channel_attention(input_feature, ratio=8):
    """Channel attention module."""
    channel = input_feature.shape[-1]
    
    # Shared dense layer for MLP
    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)
    
    # Average pooling branch
    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    avg_pool = layers.Reshape((1, 1, channel))(avg_pool)
    avg_pool = shared_dense_one(avg_pool)
    avg_pool = shared_dense_two(avg_pool)
    
    # Max pooling branch
    max_pool = layers.GlobalMaxPooling2D()(input_feature)
    max_pool = layers.Reshape((1, 1, channel))(max_pool)
    max_pool = shared_dense_one(max_pool)
    max_pool = shared_dense_two(max_pool)
    
    # Combine and sigmoid
    combined = layers.Add()([avg_pool, max_pool])
    channel_attention_map = layers.Activation('sigmoid')(combined)
    
    # Apply to input
    return layers.Multiply()([input_feature, channel_attention_map])

def spatial_attention(input_feature):
    """Spatial attention module."""
    # Average and max along channel axis
    avg_pool = layers.Lambda(lambda x: tf.reduce_mean(x, axis=3, keepdims=True))(input_feature)
    max_pool = layers.Lambda(lambda x: tf.reduce_max(x, axis=3, keepdims=True))(input_feature)
    
    # Concatenate
    concat = layers.Concatenate(axis=3)([avg_pool, max_pool])
    
    # Convolution to produce spatial attention map
    spatial_attention_map = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)
    
    # Apply to input
    return layers.Multiply()([input_feature, spatial_attention_map])

def cbam_block(input_feature, ratio=8):
    """Full CBAM: channel attention then spatial attention."""
    x = channel_attention(input_feature, ratio)
    x = spatial_attention(x)
    return x

In [79]:
# Cell 7: Model builder
def build_model(base_model_name='resnet', use_cbam=True, input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    # Select base model
    if base_model_name == 'resnet':
        base = ResNet101(weights='imagenet', include_top=False, input_shape=input_shape)
    else:  # efficientnet
        base = EfficientNetB4(weights='imagenet', include_top=False, input_shape=input_shape)
    
    # Freeze base model initially
    base.trainable = False
    
    inputs = keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    
    # Apply CBAM if requested
    if use_cbam:
        x = cbam_block(x)
    
    # Global pooling and classifier head
    # x = layers.GlobalAveragePooling2D()(x)
    x = layers.Flatten()(x)
    # x = layers.Dropout(0.25)(x)
    x = layers.Dense(128, activation='selu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(56, activation='tanh')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(16, activation='tanh')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs, outputs)
    return model, base

# Instantiate model (change base_model_name to 'resnet' if desired)
# model, base_model = build_model(base_model_name='efficientnet', use_cbam=True)
model, base_model = build_model(base_model_name='resnet', use_cbam=True)
model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_11      │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet101           │ (None, 7, 7,      │ 42,658,176 │ input_layer_11[0… │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet101[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 2048)      │          0 │ resnet101[0][0]   │
│ (GlobalMaxPooling2… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_10          │ (None, 1, 1,      │          0 │ global_average_p… │
│ (Reshape)           │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_11          │ (None, 1, 1,      │          0 │ global_max_pooli… │
│ (Reshape)           │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_21 (Dense)    │ (None, 1, 1, 256) │    524,544 │ reshape_10[0][0], │
│                     │                   │            │ reshape_11[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_22 (Dense)    │ (None, 1, 1,      │    526,336 │ dense_21[0][0],   │
│                     │ 2048)             │            │ dense_21[1][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 1, 1,      │          0 │ dense_22[0][0],   │
│                     │ 2048)             │            │ dense_22[1][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_5        │ (None, 1, 1,      │          0 │ add_5[0][0]       │
│ (Activation)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_10         │ (None, 7, 7,      │          0 │ resnet101[0][0],  │
│ (Multiply)          │ 2048)             │            │ activation_5[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_10 (Lambda)  │ (None, 7, 7, 1)   │          0 │ multiply_10[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_11 (Lambda)  │ (None, 7, 7, 1)   │          0 │ multiply_10[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_5       │ (None, 7, 7, 2)   │          0 │ lambda_10[0][0],  │
│ (Concatenate)       │                   │            │ lambda_11[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 7, 7, 1)   │         99 │ concatenate_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_11         │ (None, 7, 7,      │          0 │ multiply_10[0][0… │
│ (Multiply)          │ 2048)             │            │ conv2d_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 100352)    │          0 │ multiply_11[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_23 (Dense)    │ (None, 128)       │ 12,845,184 │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 56,562,492 (215.77 MB)

 Trainable params: 13,904,316 (53.04 MB)

 Non-trainable params: 42,658,176 (162.73 MB)

In [80]:
# Cell 8: Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)


In [81]:
# Callbacks
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_deepfake_model3.h5', monitor='val_loss', save_best_only=True, verbose=1
)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=5, min_lr=1e-7, verbose=1
)


In [82]:
history1 = model.fit(
    train_gen,                 # <-- use train_gen, not train_generator
    steps_per_epoch=steps_per_epoch,
    epochs=70,
    validation_data=val_gen,   # <-- use val_gen
    validation_steps=validation_steps,
    callbacks=[checkpoint, early_stop, reduce_lr],
    verbose=1
)

Epoch 1/70
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 573ms/step - accuracy: 0.5060 - loss: 0.7617 - precision_8: 0.4224 - recall_8: 0.3289
Epoch 1: val_loss improved from inf to 0.71982, saving model to best_deepfake_model3.h5


29/29 ━━━━━━━━━━━━━━━━━━━━ 51s 1s/step - accuracy: 0.5062 - loss: 0.7615 - precision_8: 0.4236 - recall_8: 0.3314 - val_accuracy: 0.4560 - val_loss: 0.7198 - val_precision_8: 0.4560 - val_recall_8: 1.0000 - learning_rate: 0.0010
Epoch 2/70
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 340ms/step - accuracy: 0.4481 - loss: 0.7911 - precision_8: 0.3882 - recall_8: 0.4658
Epoch 2: val_loss improved from 0.71982 to 0.71253, saving model to best_deepfake_model3.h5


29/29 ━━━━━━━━━━━━━━━━━━━━ 12s 417ms/step - accuracy: 0.4489 - loss: 0.7902 - precision_8: 0.3893 - recall_8: 0.4643 - val_accuracy: 0.4560 - val_loss: 0.7125 - val_precision_8: 0.4560 - val_recall_8: 1.0000 - learning_rate: 0.0010
Epoch 3/70
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 356ms/step - accuracy: 0.5052 - loss: 0.7316 - precision_8: 0.4550 - recall_8: 0.4558
Epoch 3: val_loss improved from 0.71253 to 0.69444, saving model to best_deepfake_model3.h5


29/29 ━━━━━━━━━━━━━━━━━━━━ 13s 432ms/step - accuracy: 0.5060 - loss: 0.7312 - precision_8: 0.4559 - recall_8: 0.4533 - val_accuracy: 0.5440 - val_loss: 0.6944 - val_precision_8: 0.0000e+00 - val_recall_8: 0.0000e+00 - learning_rate: 0.0010
Epoch 4/70
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 350ms/step - accuracy: 0.5365 - loss: 0.7223 - precision_8: 0.5270 - recall_8: 0.4466
Epoch 4: val_loss did not improve from 0.69444
29/29 ━━━━━━━━━━━━━━━━━━━━ 11s 373ms/step - accuracy: 0.5364 - loss: 0.7224 - precision_8: 0.5257 - recall_8: 0.4462 - val_accuracy: 0.5440 - val_loss: 0.7262 - val_precision_8: 0.0000e+00 - val_recall_8: 0.0000e+00 - learning_rate: 0.0010
Epoch 5/70
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 349ms/step - accuracy: 0.5169 - loss: 0.7121 - precision_8: 0.4727 - recall_8: 0.3440
Epoch 5: val_loss improved from 0.69444 to 0.68886, saving model to best_deepfake_model3.h5


29/29 ━━━━━━━━━━━━━━━━━━━━ 12s 427ms/step - accuracy: 0.5167 - loss: 0.7123 - precision_8: 0.4720 - recall_8: 0.3437 - val_accuracy: 0.5440 - val_loss: 0.6889 - val_precision_8: 0.0000e+00 - val_recall_8: 0.0000e+00 - learning_rate: 0.0010
Epoch 6/70
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 344ms/step - accuracy: 0.4979 - loss: 0.7278 - precision_8: 0.4262 - recall_8: 0.3752
Epoch 6: val_loss did not improve from 0.68886
29/29 ━━━━━━━━━━━━━━━━━━━━ 11s 365ms/step - accuracy: 0.4981 - loss: 0.7274 - precision_8: 0.4269 - recall_8: 0.3750 - val_accuracy: 0.5389 - val_loss: 0.6905 - val_precision_8: 0.3333 - val_recall_8: 0.0114 - learning_rate: 0.0010
Epoch 7/70
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 356ms/step - accuracy: 0.5416 - loss: 0.7014 - precision_8: 0.5097 - recall_8: 0.4103
Epoch 7: val_loss did not improve from 0.68886
29/29 ━━━━━━━━━━━━━━━━━━━━ 11s 378ms/step - accuracy: 0.5412 - loss: 0.7014 - precision_8: 0.5087 - recall_8: 0.4083 - val_accuracy: 0.4560 - val_loss: 0.7025 - val_precision_8: 0.

29/29 ━━━━━━━━━━━━━━━━━━━━ 12s 421ms/step - accuracy: 0.5280 - loss: 0.7014 - precision_8: 0.4743 - recall_8: 0.4680 - val_accuracy: 0.5440 - val_loss: 0.6835 - val_precision_8: 0.0000e+00 - val_recall_8: 0.0000e+00 - learning_rate: 0.0010
Epoch 9/70
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 342ms/step - accuracy: 0.5594 - loss: 0.6921 - precision_8: 0.5405 - recall_8: 0.4712
Epoch 9: val_loss did not improve from 0.68351
29/29 ━━━━━━━━━━━━━━━━━━━━ 11s 364ms/step - accuracy: 0.5592 - loss: 0.6920 - precision_8: 0.5395 - recall_8: 0.4706 - val_accuracy: 0.5440 - val_loss: 0.6876 - val_precision_8: 0.0000e+00 - val_recall_8: 0.0000e+00 - learning_rate: 0.0010
Epoch 10/70
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 347ms/step - accuracy: 0.5119 - loss: 0.7091 - precision_8: 0.4719 - recall_8: 0.3347
Epoch 10: val_loss improved from 0.68351 to 0.68191, saving model to best_deepfake_model3.h5


29/29 ━━━━━━━━━━━━━━━━━━━━ 12s 421ms/step - accuracy: 0.5125 - loss: 0.7090 - precision_8: 0.4722 - recall_8: 0.3355 - val_accuracy: 0.5440 - val_loss: 0.6819 - val_precision_8: 0.0000e+00 - val_recall_8: 0.0000e+00 - learning_rate: 0.0010
Epoch 11/70
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 349ms/step - accuracy: 0.5383 - loss: 0.7034 - precision_8: 0.4527 - recall_8: 0.2616
Epoch 11: val_loss did not improve from 0.68191
29/29 ━━━━━━━━━━━━━━━━━━━━ 11s 371ms/step - accuracy: 0.5376 - loss: 0.7038 - precision_8: 0.4524 - recall_8: 0.2605 - val_accuracy: 0.5492 - val_loss: 0.6918 - val_precision_8: 0.5034 - val_recall_8: 0.8523 - learning_rate: 0.0010
Epoch 12/70
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 338ms/step - accuracy: 0.5613 - loss: 0.6946 - precision_8: 0.5149 - recall_8: 0.4717
Epoch 12: val_loss did not improve from 0.68191
29/29 ━━━━━━━━━━━━━━━━━━━━ 11s 360ms/step - accuracy: 0.5609 - loss: 0.6948 - precision_8: 0.5148 - recall_8: 0.4692 - val_accuracy: 0.5440 - val_loss: 0.6900 - val_precision_8

In [73]:
# Fine-tune: unfreeze base and continue training
print("\nUnfreezing base model for fine-tuning...")
base_model.trainable = True
# Optionally freeze early layers (ResNet) – for EfficientNet, fine-tuning all is okay with small LR
if base_model.name == 'resnet50':
    # Freeze first 100 layers for ResNet
    for layer in base_model.layers[:100]:
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # lower LR
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)


history2 = model.fit(
    train_gen,                 # <-- use train_gen, not train_generator
    steps_per_epoch=steps_per_epoch,
    epochs=20,
    validation_data=val_gen,   # <-- use val_gen
    validation_steps=validation_steps,
    callbacks=[checkpoint, early_stop, reduce_lr],
    verbose=1
)


Unfreezing base model for fine-tuning...
Epoch 1/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4131 - loss: 12.5861 - precision_6: 0.4097 - recall_6: 1.0000
Epoch 1: val_loss did not improve from 0.56023
29/29 ━━━━━━━━━━━━━━━━━━━━ 163s 2s/step - accuracy: 0.4156 - loss: 12.4524 - precision_6: 0.4118 - recall_6: 1.0000 - val_accuracy: 0.4560 - val_loss: 23.6166 - val_precision_6: 0.4560 - val_recall_6: 1.0000 - learning_rate: 1.0000e-05
Epoch 2/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 379ms/step - accuracy: 0.7290 - loss: 1.9781 - precision_6: 0.6229 - recall_6: 0.9994
Epoch 2: val_loss did not improve from 0.56023
29/29 ━━━━━━━━━━━━━━━━━━━━ 12s 400ms/step - accuracy: 0.7326 - loss: 1.9484 - precision_6: 0.6268 - recall_6: 0.9994 - val_accuracy: 0.4560 - val_loss: 74.0270 - val_precision_6: 0.4560 - val_recall_6: 1.0000 - learning_rate: 1.0000e-05
Epoch 3/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 370ms/step - accuracy: 0.9690 - loss: 0.1880 - precision_6: 0.9438 - recall_6: 0.9942
Epoch 3: val

KeyboardInterrupt: 

In [83]:
# =============================================================================
# Tex-ViT: Texture-Enhanced ResNet + ViT with Cross-Attention (TensorFlow)
# Based on the paper by Dagar & Vishwakarma (2024) 
# =============================================================================

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
import numpy as np

# =============================================================================
# 1. GRAM MATRIX FOR TEXTURE FEATURES
# =============================================================================

class GramMatrix(layers.Layer):
    """
    Computes Gram matrix for texture feature extraction.
    Gram matrix captures correlations between feature channels,
    which represent texture patterns in the image [citation:1].
    """
    def __init__(self, **kwargs):
        super(GramMatrix, self).__init__(**kwargs)
    
    def call(self, features):
        """
        Args:
            features: Tensor of shape (batch, height, width, channels)
        Returns:
            gram: Tensor of shape (batch, channels, channels)
        """
        batch_size, h, w, channels = tf.shape(features)[0], tf.shape(features)[1], tf.shape(features)[2], tf.shape(features)[3]
        
        # Reshape to (batch, channels, height*width)
        features_reshaped = tf.reshape(features, [batch_size, h * w, channels])
        
        # Compute Gram matrix: features * features^T
        gram = tf.matmul(features_reshaped, features_reshaped, transpose_a=True)
        
        # Normalize
        gram = gram / tf.cast(h * w * channels, tf.float32)
        
        return gram



In [84]:
# =============================================================================
# 2. TEXTURE MODULE (Multi-scale Gram Extraction)
# =============================================================================

class TextureModule(layers.Layer):
    """
    Extracts texture features using Gram matrices at multiple scales.
    This module operates on ResNet feature maps before downsampling [citation:2].
    """
    def __init__(self, filters, **kwargs):
        super(TextureModule, self).__init__(**kwargs)
        self.filters = filters
        
        # Projection layer to reduce dimensions before Gram computation
        self.texture_projection = layers.Conv2D(
            filters // 2, 
            kernel_size=1,
            padding='same',
            name=f'texture_proj_{filters}'
        )
        self.bn = layers.BatchNormalization()
        self.relu = layers.ReLU()
        self.gram = GramMatrix()
        
    def call(self, x, training=False):
        """
        Args:
            x: Input feature map from ResNet stage
        Returns:
            gram_features: Flattened Gram matrix (texture representation)
            texture_feat: Projected feature maps (for visualization)
        """
        # Project to reduce dimensions
        texture_feat = self.texture_projection(x)
        texture_feat = self.bn(texture_feat, training=training)
        texture_feat = self.relu(texture_feat)
        
        # Compute Gram matrix for texture correlation
        gram_features = self.gram(texture_feat)  # [batch, channels/2, channels/2]
        
        # Flatten Gram matrix for feeding into transformer
        batch_size = tf.shape(gram_features)[0]
        gram_flat = tf.reshape(gram_features, [batch_size, -1])
        
        return gram_flat, texture_feat



In [85]:
# =============================================================================
# 3. CROSS-ATTENTION FUSION LAYER
# =============================================================================

class CrossAttentionFusion(layers.Layer):
    """
    Dual-branch cross-attention between texture features and ViT features.
    This is the core innovation of Tex-ViT [citation:1].
    """
    def __init__(self, embed_dim, num_heads=8, **kwargs):
        super(CrossAttentionFusion, self).__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        
        # Multi-head attention for cross-attention
        self.mha = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            name='cross_attention'
        )
        
        # Layer normalization
        self.layernorm_query = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm_key = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm_out = layers.LayerNormalization(epsilon=1e-6)
        
        # Feed-forward network after attention
        self.ffn = keras.Sequential([
            layers.Dense(embed_dim * 4, activation='gelu'),
            layers.Dense(embed_dim)
        ], name='cross_attention_ffn')
        
    def call(self, query, key_value, training=False):
        """
        Args:
            query: ViT features [batch, seq_len, embed_dim]
            key_value: Texture features [batch, 1, texture_dim] (expanded)
        Returns:
            attended_features: [batch, seq_len, embed_dim]
        """
        # Apply layer norm
        query_norm = self.layernorm_query(query)
        key_norm = self.layernorm_key(key_value)
        
        # Cross-attention: query attends to key_value
        attended, attention_scores = self.mha(
            query=query_norm,
            key=key_norm,
            value=key_norm,
            return_attention_scores=True,
            training=training
        )
        
        # Residual connection + layer norm
        attended = query + attended
        attended = self.layernorm_out(attended)
        
        # FFN with residual
        ffn_output = self.ffn(attended)
        output = attended + ffn_output
        
        return output, attention_scores



In [86]:
# =============================================================================
# 4. MAIN TEX-VIT MODEL
# =============================================================================

def TexViT(
    input_shape=(224, 224, 3),
    num_classes=1,
    pretrained=True
):
    """
    Tex-ViT: Texture-enhanced ResNet + Vision Transformer with cross-attention.
    
    Architecture:
    1. ResNet backbone (ResNet18/50) extracts multi-scale features
    2. Texture modules compute Gram matrices at multiple scales
    3. Vision Transformer processes global features
    4. Cross-attention fuses texture and ViT features
    5. Classification head [citation:2][citation:4]
    
    Args:
        input_shape: Input image dimensions
        num_classes: Number of output classes (1 for binary)
        pretrained: Use pretrained ImageNet weights
    
    Returns:
        Keras Model
    """
    
    # Input layer
    inputs = layers.Input(shape=input_shape, name='input_image')
    
    # =========================================================================
    # 4.1 RESNET BACKBONE (Multi-scale Feature Extraction)
    # =========================================================================
    # Using ResNet50 as the backbone (paper uses ResNet18, but 50 works better)
    base_model = keras.applications.ResNet50(
        include_top=False,
        weights='imagenet' if pretrained else None,
        input_tensor=inputs
    )
    
    # Extract intermediate layer outputs for texture modules
    # Get outputs at different stages (before each downsampling) [citation:1]
    layer_outputs = {
        'stage1': base_model.get_layer('conv1_relu').output,      # After first conv
        'stage2': base_model.get_layer('conv2_block3_out').output, # After stage 2
        'stage3': base_model.get_layer('conv3_block4_out').output, # After stage 3
        'stage4': base_model.get_layer('conv4_block6_out').output, # After stage 4
    }
    
    # Create feature extraction model
    feature_extractor = Model(
        inputs=base_model.inputs,
        outputs=layer_outputs,
        name='resnet_feature_extractor'
    )
    
    # Extract features at multiple scales
    features = feature_extractor(inputs)
    
    # =========================================================================
    # 4.2 TEXTURE MODULES (Parallel to ResNet stages)
    # =========================================================================
    texture_features = []
    texture_dims = []
    
    # Texture module for stage 2 (64 channels)
    gram2, feat2 = TextureModule(64, name='texture_stage2')(features['stage2'])
    texture_features.append(gram2)
    texture_dims.append(gram2.shape[-1])
    
    # Texture module for stage 3 (128 channels)
    gram3, feat3 = TextureModule(128, name='texture_stage3')(features['stage3'])
    texture_features.append(gram3)
    texture_dims.append(gram3.shape[-1])
    
    # Texture module for stage 4 (256 channels)
    gram4, feat4 = TextureModule(256, name='texture_stage4')(features['stage4'])
    texture_features.append(gram4)
    texture_dims.append(gram4.shape[-1])
    
    # Concatenate all texture Gram features
    combined_texture = layers.Concatenate(axis=-1, name='concat_texture')(texture_features)
    
    # Project texture features to match ViT embedding dimension
    texture_projection = layers.Dense(768, activation='gelu', name='texture_projection')(combined_texture)
    texture_projection = layers.Dropout(0.1)(texture_projection)
    
    # Expand dimensions for cross-attention (add sequence dimension)
    texture_expanded = tf.expand_dims(texture_projection, axis=1)  # [batch, 1, 768]
    
    # =========================================================================
    # 4.3 VISION TRANSFORMER (Global Feature Processing)
    # =========================================================================
    
    # Use a pre-trained Vision Transformer
    # Since Keras doesn't have built-in ViT, we'll use the one from KerasCV
    # If keras-cv is not installed, we'll implement a minimal ViT
    
    try:
        import keras_cv
        vit_encoder = keras_cv.models.ViTBackbone.from_preset(
            "vit_base_patch16_224",
            load_weights=pretrained
        )
        vit_output = vit_encoder(inputs)
        vit_features = vit_output  # [batch, 197, 768]
        
    except ImportError:
        print("keras-cv not found. Using minimal ViT implementation.")
        # Minimal ViT implementation
        patch_size = 16
        num_patches = (input_shape[0] // patch_size) ** 2
        
        # Patch embedding
        patches = layers.Conv2D(
            768, kernel_size=patch_size, strides=patch_size, 
            padding='valid', name='patch_embed'
        )(inputs)
        patches = tf.reshape(patches, [-1, num_patches, 768])
        
        # Add position embeddings
        positions = tf.range(start=0, limit=num_patches, delta=1)
        pos_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=768, name='position_embedding'
        )(positions)
        vit_features = patches + pos_embedding
        
        # Add CLS token
        cls_token = tf.Variable(tf.random.normal([1, 1, 768]), trainable=True, name='cls_token')
        cls_tokens = tf.tile(cls_token, [tf.shape(vit_features)[0], 1, 1])
        vit_features = tf.concat([cls_tokens, vit_features], axis=1)
        
        # Transformer layers (simplified - 6 layers instead of 12)
        for i in range(6):
            vit_features = transformer_block(vit_features, 768, 8, name=f'transformer_{i}')
    
    # Extract CLS token for classification
    vit_cls = vit_features[:, 0, :]  # [batch, 768]
    vit_cls_expanded = tf.expand_dims(vit_cls, axis=1)  # [batch, 1, 768]
    
    # =========================================================================
    # 4.4 CROSS-ATTENTION FUSION
    # =========================================================================
    
    # Cross-attention: ViT CLS attends to texture features [citation:1]
    cross_attention = CrossAttentionFusion(embed_dim=768, num_heads=8, name='cross_attention')
    attended_features, attention_weights = cross_attention(
        query=vit_cls_expanded,
        key_value=texture_expanded,
        training=True
    )
    
    # Squeeze the sequence dimension
    attended_features = tf.squeeze(attended_features, axis=1)  # [batch, 768]
    
    # =========================================================================
    # 4.5 CLASSIFICATION HEAD
    # =========================================================================
    
    # Combine original CLS and attended features (residual connection)
    combined_features = layers.Concatenate(axis=-1, name='concat_vit_texture')([
        vit_cls,
        attended_features
    ])  # [batch, 1536]
    
    # Classification layers
    x = layers.Dense(512, activation='relu', name='fc1')(combined_features)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Dense(128, activation='relu', name='fc2')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    # Output layer (sigmoid for binary classification)
    outputs = layers.Dense(num_classes, activation='sigmoid', name='output')(x)
    
    # Create model
    model = Model(inputs=inputs, outputs=outputs, name='TexViT')
    
    return model


In [87]:
# =============================================================================
# 5. TRANSFORMER BLOCK (for minimal ViT implementation)
# =============================================================================

def transformer_block(x, embed_dim, num_heads, name=None):
    """Single transformer encoder block"""
    
    # Multi-head self-attention
    attn_output = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=embed_dim // num_heads,
        name=f'{name}_mha'
    )(x, x)
    
    # Add & Norm
    x = layers.LayerNormalization(epsilon=1e-6)(x + attn_output)
    
    # Feed-forward network
    ffn_output = keras.Sequential([
        layers.Dense(embed_dim * 4, activation='gelu'),
        layers.Dense(embed_dim)
    ], name=f'{name}_ffn')(x)
    
    # Add & Norm
    x = layers.LayerNormalization(epsilon=1e-6)(x + ffn_output)
    
    return x




In [88]:
# =============================================================================
# 6. MODEL INSTANTIATION AND COMPILATION
# =============================================================================

def create_texvit_model(
    input_shape=(224, 224, 3),
    num_classes=1,
    learning_rate=1e-4
):
    """
    Create and compile Tex-ViT model
    """
    model = TexViT(
        input_shape=input_shape,
        num_classes=num_classes,
        pretrained=True
    )
    
    # Compile with binary crossentropy for binary classification
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
            keras.metrics.AUC(name='auc')
        ]
    )
    
    model.summary()
    return model



In [89]:
# =============================================================================
# 7. CUSTOM CALLBACK FOR TEXTURE VISUALIZATION (Optional)
# =============================================================================

class TextureVisualizationCallback(keras.callbacks.Callback):
    """
    Callback to visualize Gram matrix features during training
    Helps with debugging and explainability
    """
    def __init__(self, validation_data, log_dir='./logs'):
        super().__init__()
        self.validation_data = validation_data
        self.log_dir = log_dir
        self.file_writer = tf.summary.create_file_writer(log_dir)
        
    def on_epoch_end(self, epoch, logs=None):
        # Get a batch of validation data
        x_val, y_val = next(iter(self.validation_data))
        
        # Create a submodel to extract intermediate features
        # This requires building the model with specific layer outputs
        # Implementation depends on your specific needs
        
        # Log metrics
        with self.file_writer.as_default():
            for key, value in logs.items():
                tf.summary.scalar(key, value, step=epoch)



In [92]:
# =============================================================================
# 8. TRAINING LOOP INTEGRATION WITH YOUR EXISTING DATA
# =============================================================================

def train_texvit_with_your_data(model, train_gen, val_gen, epochs=20):
    """
    Train Tex-ViT using your existing data generators
    """
    
    # Callbacks
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            'best_texvit_model.h5',
            monitor='val_accuracy',
            save_best_only=True,
            mode='max',
            verbose=1
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=7,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        ),
        keras.callbacks.TensorBoard(
            log_dir='./texvit_logs',
            histogram_freq=1,
            write_graph=True
        )
    ]
    
    # Train
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )
    
    return history



In [93]:
# =============================================================================
# 9. EXAMPLE USAGE WITH YOUR DATA
# =============================================================================

# Example showing how to integrate with your existing data pipeline
if __name__ == "__main__":
    
    # Assume you already have:
    # train_gen, val_gen, test_gen from your previous code
    
    # Create model
    model = create_texvit_model(
        input_shape=(224, 224, 3),
        num_classes=1,
        learning_rate=1e-4
    )
    
    # First phase: Train with frozen ResNet backbone
    print("Phase 1: Training with frozen ResNet backbone...")
    
    # Freeze all ResNet layers
    for layer in model.layers:
        if 'resnet' in layer.name.lower() or isinstance(layer, keras.Model) and 'resnet' in layer.name.lower():
            layer.trainable = False
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall(), keras.metrics.AUC()]
    )
    
    history1 = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=10,
        callbacks=[
            keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
        ],
        verbose=1
    )
    

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


ValueError: Shapes used to initialize variables must be fully-defined (no `None` dimensions). Received: shape=(None, 768) for variable path='texture_projection/kernel'

In [ ]:





    # Second phase: Fine-tune entire model
    print("\nPhase 2: Fine-tuning entire Tex-ViT...")
    
    # Unfreeze all layers
    for layer in model.layers:
        layer.trainable = True
    
    # Lower learning rate for fine-tuning
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall(), keras.metrics.AUC()]
    )
    
    history2 = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=15,
        callbacks=[
            keras.callbacks.ModelCheckpoint('best_texvit_finetuned.h5', save_best_only=True),
            keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2)
        ],
        verbose=1
    )
    
    # Evaluate on test set
    print("\nEvaluating on test set...")
    test_results = model.evaluate(test_gen)
    print(f"Test Loss: {test_results[0]:.4f}")
    print(f"Test Accuracy: {test_results[1]:.4f}")
    print(f"Test Precision: {test_results[2]:.4f}")
    print(f"Test Recall: {test_results[3]:.4f}")
    print(f"Test AUC: {test_results[4]:.4f}")